# Simple Sorter Gate Validation

Loads a pre-processed `shank_recording.bin` saved by `run_shank.py` and runs
`run_sorter('simple')` on it to check whether the gate would pass or stop.

Also validates that a synthetic silent recording correctly returns 0 units.

In [ ]:
from pathlib import Path
import numpy as np
import probeinterface as pi
import spikeinterface.full as si
import spikeinterface.extractors as se
from kilosort import io as ks_io

## Configuration — edit these

In [ ]:
FOLDER        = Path('/groups/voigts/voigtslab/neuropixels_2025/npx10/2025_12_18_large_maze/')  # top-level session folder
PROBE         = 'b'                               # probe letter, e.g. 'A'
SHANK_NUM     = '3'                               # shank to test


SHANK_FOLDER = FOLDER / f'output/{PROBE}/shank_{SHANK_NUM}'

SAMPLE_RATE = 30000

# Detection threshold in std (default used in run_shank.py is 5.0)
DETECT_THRESHOLD = 5.0

# --- derived paths ---
bin_file   = SHANK_FOLDER / 'shank_recording.bin'
probe_file = SHANK_FOLDER / 'probe.prb'
scratch    = SHANK_FOLDER / 'simple_sorter_check'

assert bin_file.exists(),   f'Binary not found: {bin_file}'
assert probe_file.exists(), f'Probe file not found: {probe_file}'
print('Files found OK.')

## Load the saved recording

In [ ]:
# Read probe geometry from kilosort prb file
ks_probe = ks_io.load_probe(probe_file)
n_channels = ks_probe['n_chan']
print(f'Channels: {n_channels}')

# Build a probeinterface Probe from the kilosort xc/yc
probe = pi.Probe(ndim=2)
positions = np.column_stack([ks_probe['xc'], ks_probe['yc']])
probe.set_contacts(positions=positions, shapes='circle', shape_params={'radius': 5})
probe.create_auto_shape()

# Load the binary
recording = se.read_binary(
    bin_file,
    dtype='int16',
    sampling_frequency=SAMPLE_RATE,
    num_channels=n_channels,
)
recording = recording.set_probe(probe)
recording.set_channel_gains(1)
recording.set_channel_offsets(0)

print(recording)
print(f'Duration: {recording.get_total_duration():.1f} s')

## Run simple sorter

In [ ]:
real_sort_dir = scratch / 'real'

sorting = si.run_sorter(
    sorter_name='simple',
    recording=recording,
    output_folder=real_sort_dir,
    remove_existing_folder=True,
    verbose=True,
    sorter_params={'detect_threshold': DETECT_THRESHOLD},
)

unit_ids = sorting.get_unit_ids()
print(f'\nUnits found: {len(unit_ids)}')

In [ ]:
import pandas as pd

if len(unit_ids) > 0:
    rows = []
    for uid in unit_ids:
        spikes = sorting.get_unit_spike_train(uid, segment_index=0)
        rows.append({
            'unit_id': uid,
            'n_spikes': len(spikes),
            'firing_rate_hz': len(spikes) / recording.get_total_duration(),
        })
    df = pd.DataFrame(rows).sort_values('firing_rate_hz', ascending=False)
    print(df.to_string(index=False))
    print(f'\nTotal spikes: {df["n_spikes"].sum()}')
else:
    print('Gate would STOP — 0 units found on real recording.')

## Sanity check: synthetic silent recording must return 0 units

Replace all samples with low-amplitude Gaussian noise (well below threshold)
to confirm the gate does not false-positive on dead/silent channels.

In [ ]:
from spikeinterface.core import NumpyRecording

rng = np.random.default_rng(0)
n_fr = recording.get_num_frames()
silent_traces = rng.normal(0, 1, size=(n_fr, n_channels)).astype('int16')

silent_rec = NumpyRecording(silent_traces, sampling_frequency=SAMPLE_RATE)
silent_rec = silent_rec.set_probe(probe)
silent_rec.set_channel_gains(1)
silent_rec.set_channel_offsets(0)

silent_sort_dir = scratch / 'silent'
sorting_silent = si.run_sorter(
    sorter_name='simple',
    recording=silent_rec,
    output_folder=silent_sort_dir,
    remove_existing_folder=True,
    verbose=False,
    sorter_params={'detect_threshold': DETECT_THRESHOLD},
)

n_silent = len(sorting_silent.get_unit_ids())
print(f'Silent recording → {n_silent} units')
if n_silent == 0:
    print('PASS: gate correctly stops on a silent recording.')
else:
    print('WARNING: gate would NOT stop on silent data — raise DETECT_THRESHOLD.')

## Summary

In [ ]:
decision = 'PASS → continue to Kilosort' if len(unit_ids) > 0 else 'STOP → write NO_GOOD_SPIKES'

print('=== Simple Sorter Gate Summary ===')
print(f'  Shank folder     : {SHANK_FOLDER}')
print(f'  Duration         : {recording.get_total_duration():.1f} s')
print(f'  Channels         : {n_channels}')
print(f'  Detect threshold : {DETECT_THRESHOLD} std')
print(f'  Units (real)     : {len(unit_ids)}')
print(f'  Units (silent)   : {n_silent}')
print(f'  Gate decision    : {decision}')